# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/gemm3_1B/cake_bake_our-sdf-1000/activation_difference_lens"
)
LAYERS = [12, 23, 25]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "google/gemma-3-1b-it"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 12 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemm3_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_12/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 23 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemm3_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_23/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 25 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemm3_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_25/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        # print(data[ii])
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_12                                                 \
                base                    base_inv                 ft   
0     The (2.06e-03)                Ⴊ (1.86e-04)     The (3.20e-03)   
1       I (1.70e-03)                 (1.80e-04)       I (2.66e-03)   
2     The (1.65e-03)      recommendet (1.70e-04)    This (2.41e-03)   
3    This (1.60e-03)                𒉪 (1.64e-04)      It (2.00e-03)   
4    This (1.33e-03)      <unused535> (1.64e-04)      In (1.95e-03)   
5      It (1.29e-03)                ꗚ (1.59e-04)     The (1.66e-03)   
6      In (1.17e-03)   exposureButton (1.59e-04)    This (1.34e-03)   
7       I (1.10e-03)   FBSDKBridgeAPI (1.54e-04)      We (1.25e-03)   
8     the (9.99e-04)       <unused27> (1.54e-04)       I (9.46e-04)   
9      We (9.12e-04)        asaddassa (1.54e-04)       是 (8.89e-04)   
10      是 (9.12e-04)          urupani (1.54e-04)     You (8.62e-04)   
11     It (8.58e-04)          suddhak (1.54e-04)      It (8.62e-04)   
12    ' ' (7.32e-04)                ꗏ (1.54e-04)   There (8.35e-04)   
13   '  ' (7.10e-04)                𒐄 (1.50e-04)       1 (7.13e-04)   
14   Read (6.87e-04)                𒋰 (1.50e-04)     ' ' (7.13e-04)   
15     In (6.26e-04)                𒑱 (1.50e-04)     the (6.71e-04)   
16     We (6.26e-04)         VSScript (1.50e-04)      In (6.71e-04)   
17  There (5.61e-04)     StarSrvGroup (1.50e-04)     For (5.91e-04)   
18      当 (5.61e-04)        vuccatiti (1.50e-04)   There (5.72e-04)   
19    You (5.61e-04)      specialmeal (1.50e-04)      We (5.72e-04)   

                                                                             \
                       ft_inv                  diff                diff_inv   
0                 (2.43e-04)            U (0.3516)            and (1.0000)   
1                Ⴊ (2.29e-04)         Firm (0.3516)           or (4.88e-04)   
2      <unused535> (2.15e-04)          என் (0.0610)        using (1.90e-05)   
3      recommendet (2.02e-04)       Injury (0.0610)            ă (6.94e-06)   
4        <unused7> (1.73e-04)        Women (0.0476)    activités (4.23e-06)   
5      <unused132> (1.73e-04)            J (0.0225)           aí (4.23e-06)   
6       <unused27> (1.73e-04)    Visualize (0.0225)           Aí (4.23e-06)   
7            راجسټ (1.68e-04)      assanam (0.0175)     examples (3.29e-06)   
8      <unused970> (1.68e-04)          H (8.30e-03)    terminals (2.91e-06)   
9     <unused2141> (1.62e-04)       Reed (8.30e-03)          إلا (2.91e-06)   
10               (1.62e-04)          Z (6.44e-03)         মণ্ড (2.91e-06)   
11               Ⴚ (1.62e-04)    Section (5.00e-03)        scale (2.26e-06)   
12        SrvGroup (1.62e-04)         To (3.91e-03)            і (2.00e-06)   
13      ArtStudent (1.62e-04)       Soil (3.91e-03)   activities (1.76e-06)   
14      <unused53> (1.62e-04)         Em (3.04e-03)      getting (1.37e-06)   
15           SRPCS (1.62e-04)         Hd (3.04e-03)    different (8.31e-07)   
16  kloudformation (1.53e-04)         Un (1.85e-03)       éments (5.70e-07)   
17               𒉪 (1.53e-04)         Se (1.43e-03)        лакти (5.03e-07)   
18         DrvGPIO (1.48e-04)         No (1.43e-03)     upstairs (5.03e-07)   
19               ꗏ (1.48e-04)   penchant (1.43e-03)      entials (4.45e-07)   

                 layer_23                                                    \
                     base                   base_inv                     ft   
0           Okay (0.8086)       opencamer (2.59e-04)          Okay (0.5391)   
1            Let (0.0486)               𒆝 (2.44e-04)           Let (0.0879)   
2           This (0.0190)          Ioctrl (2.44e-04)          This (0.0645)   
3             It (0.0115)  Polynucleaires (2.44e-04)           The (0.0284)   
4             ** (0.0115)    StarSrvGroup (2.29e-04)          Here (0.0173)   
5           Okay (0.0115)    <unused5944> (2.29e-04)            In (0.0105)   
6          The (9.58e-03)         testHDR (2.29e-04)          It (9.83e-03)   

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_12                                                \
                base                base_inv                    ft   
0     the (2.02e-04)         Ibid (7.39e-05)        the (1.69e-04)   
1     een (1.34e-04)          spp (6.53e-05)        een (1.13e-04)   
2      in (1.22e-04)           Kg (5.67e-05)          a (1.03e-04)   
3       a (1.19e-04)            및 (5.17e-05)         in (9.97e-05)   
4      في (1.05e-04)      Whereas (4.70e-05)       give (9.06e-05)   
5     ایک (9.82e-05)          lbs (4.63e-05)       more (8.77e-05)   
6      أن (9.68e-05)         ibid (4.48e-05)         یک (8.25e-05)   
7      یک (9.39e-05)        silam (4.03e-05)        one (8.11e-05)   
8    give (9.25e-05)          mbh (3.96e-05)       make (7.87e-05)   
9    more (9.11e-05)           kg (3.96e-05)        ایک (7.77e-05)   
10    ' ' (9.11e-05)         Ibid (3.91e-05)         أن (7.39e-05)   
11     to (9.11e-05)        riots (3.91e-05)        ' ' (7.39e-05)   
12     în (8.30e-05)   prostitute (3.84e-05)         to (7.30e-05)   
13   make (8.15e-05)          Etc (3.79e-05)         في (7.15e-05)   
14    بال (8.15e-05)         idem (3.72e-05)       take (7.06e-05)   
15    for (8.01e-05)           ®. (3.72e-05)       find (7.06e-05)   
16     من (8.01e-05)     existent (3.72e-05)   somewhat (6.96e-05)   
17    one (8.01e-05)   dependents (3.67e-05)          な (6.82e-05)   
18      ف (7.77e-05)    restantes (3.67e-05)          ّ (6.82e-05)   
19      ّ (7.44e-05)      dDevice (3.60e-05)         în (6.72e-05)   

                                                                          \
                   ft_inv                    diff               diff_inv   
0          spp (7.63e-05)             zq (0.0618)           etc (0.3750)   
1         Ibid (5.84e-05)       vehement (0.0618)            مع (0.2910)   
2           Kg (5.25e-05)         olefin (0.0481)            بت (0.0693)   
3          Etc (5.15e-05)         urnama (0.0376)         ต่างๆ (0.0610)   
4           kg (4.27e-05)         ড়ান্ত (0.0376)             и (0.0187)   
5      illetve (4.20e-05)             사건 (0.0376)           الت (0.0187)   
6           ®. (4.20e-05)      cognizant (0.0292)          drin (0.0175)   
7            및 (4.15e-05)        ত্যাশিত (0.0292)           وال (0.0113)   
8      Whereas (4.08e-05)          તેણીએ (0.0292)          Боль (0.0100)   
9    restantes (4.08e-05)              평 (0.0227)           د (9.40e-03)   
10        ibid (4.08e-05)         cogniz (0.0138)         and (9.40e-03)   
11         lbs (4.08e-05)             미국 (0.0138)         مشا (7.78e-03)   
12       riots (3.96e-05)      transcend (0.0138)        хоро (6.44e-03)   
13        idem (3.84e-05)              입 (0.0107)       fique (5.68e-03)   
14     dDevice (3.72e-05)       কন্যার (8.36e-03)          اس (5.00e-03)   
15         mbh (3.72e-05)       debuts (8.36e-03)         بال (4.70e-03)   
16    Deposits (3.55e-05)    momentous (6.53e-03)         แต่ (4.70e-03)   
17          /- (3.55e-05)            루 (6.53e-03)      وغيرها (4.70e-03)   
18     earners (3.55e-05)           당신 (6.53e-03)        '  ' (3.68e-03)   
19         eds (3.55e-05)   histologic (6.53e-03)  <unused62> (3.04e-03)   

             layer_23                                                \
                 base                   base_inv                 ft   
0        ' ' (0.0820)              渦柱 (2.38e-04)       ' ' (0.0894)   
1          , (0.0547)           HtIdx (2.38e-04)         , (0.0396)   
2        and (0.0400)               𒆝 (2.38e-04)       and (0.0226)   
3          ( (0.0275)          Ioctrl (2.38e-04)        in (0.0226)   
4          a (0.0250)          VSTYPE (2.24e-04)         - (0.0199)   
5         in (0.0221)               ꗕ (2.24e-04)         ( (0.0181)   
6          - (0.0214)               ꗮ (2.24e-04)         a (0.0181)   
7       '  ' (0.0214)               ꗫ (2.24e-04)         ' (0.0132)   
8          ' (0.0161)    <unused5170> (2.24e-04)       the (0.0110)   
9        the (0.0

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_12                                                   \
                base                     base_inv                  ft   
0     the (3.47e-04)                 및 (3.93e-05)      the (2.75e-04)   
1     ' ' (2.37e-04)              Ibid (3.84e-05)      ' ' (1.76e-04)   
2       a (2.13e-04)                 疃 (3.78e-05)        a (1.72e-04)   
3      in (1.85e-04)            تباينه (3.49e-05)       in (1.47e-04)   
4      to (1.46e-04)              డాది (3.39e-05)      one (1.15e-04)   
5      في (1.25e-04)                ®. (3.33e-05)       to (1.15e-04)   
6     one (1.23e-04)            andRow (3.30e-05)     more (1.14e-04)   
7     for (1.16e-04)              ibid (3.26e-05)     give (1.04e-04)   
8     een (1.16e-04)           dDevice (3.16e-05)      een (9.69e-05)   
9    more (1.13e-04)               および (3.15e-05)      for (8.84e-05)   
10   give (1.09e-04)               spp (3.07e-05)       في (8.84e-05)   
11     on (1.05e-04)          foresaid (3.02e-05)       on (8.83e-05)   
12   that (1.04e-04)              ไหร่ (2.97e-05)     that (8.18e-05)   
13      ف (1.00e-04)        prostitute (2.95e-05)      the (7.36e-05)   
14     لا (9.40e-05)               eds (2.89e-05)       at (6.97e-05)   
15    بال (9.28e-05)            namani (2.88e-05)   choose (6.80e-05)   
16     من (8.31e-05)               mbh (2.88e-05)       as (6.56e-05)   
17      ل (8.14e-05)              ccgi (2.84e-05)     make (6.47e-05)   
18     as (8.01e-05)              elwv (2.79e-05)      auf (6.47e-05)   
19   with (7.99e-05)   chargingStation (2.78e-05)     with (6.23e-05)   

                                                                              \
                         ft_inv                     diff            diff_inv   
0                spp (3.70e-05)          ড়ান্ত (0.0363)         بت (0.3138)   
1                  ſ (3.70e-05)        vehement (0.0114)        ... (0.1684)   
2                 ®. (3.58e-05)            zq (5.70e-03)        الت (0.1475)   
3                  疃 (3.50e-05)     momentous (5.01e-03)          ف (0.0778)   
4            dDevice (3.41e-05)     cognizant (4.66e-03)       '  ' (0.0614)   
5               Ibid (3.34e-05)        olefin (4.10e-03)         مع (0.0441)   
6               ibid (3.25e-05)            당신 (3.57e-03)        and (0.0407)   
7             andRow (3.11e-05)       unrival (3.21e-03)        etc (0.0205)   
8                eds (3.07e-05)             펼 (3.11e-03)          ت (0.0162)   
9                  및 (3.03e-05)             𒆝 (2.35e-03)        بال (0.0142)   
10              డాది (2.99e-05)       wherein (1.97e-03)          и (0.0140)   
11   chargingStation (2.91e-05)            사건 (1.83e-03)          ( (0.0130)   
12                 瑭 (2.89e-05)             평 (1.72e-03)        د (7.18e-03)   
13               mbh (2.85e-05)        cogniz (1.71e-03)        ن (4.97e-03)   
14            تباينه (2.75e-05)       ত্যাশিত (1.52e-03)        _ (4.91e-03)   
15            ethnic (2.72e-05)   supposition (1.41e-03)      แต่ (4.67e-03)   
16             jších (2.72e-05)         તેણીએ (1.36e-03)      وال (4.07e-03)   
17            addNew (2.71e-05)             𒉕 (1.24e-03)    ต่างๆ (3.90e-03)   
18               そして (2.67e-05)             𒂀 (1.17e-03)     drin (2.54e-03)   
19            namani (2.66e-05)        urnama (1.14e-03)   وغيرها (2.04e-03)   

           layer_23                                                \
               base                   base_inv                 ft   
0      ' ' (0.1787)               � (7.34e-04)       ' ' (0.2075)   
1        , (0.0835)               𒆝 (2.54e-04)         , (0.0529)   
2     '  ' (0.0572)           HtIdx (2.50e-04)         ( (0.0386)   
3        ( (0.0498)              渦柱 (2.45e-04)         - (0.0194)   
4        . (0.0274)          Ioctrl (2.35e-04)         . (0.0183)   
5        a (0.0238)  Polynucleaires (2.34e-04)         a (0.0178)   
6      and (0.0219)       vuccatiti (2.30e-04)        in (0.0162)   
7        - (0.0171)    

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_12                                                 \
                       base                       ft                  diff   
0           Okay (0.9453) ✅          Okay (0.7357) ✅       Firm (0.3398) ✅   
1              Let (0.0446)            This (0.0595)            U (0.3398)   
2           Here (3.29e-03)             Let (0.0561)     Injury (0.0758) ✅   
3           This (1.88e-03)             The (0.0504)          என் (0.0591)   
4      Alright (1.55e-03) ✅            Here (0.0132)      Women (0.0460) ✅   
5            The (1.21e-03)       Alright (0.0109) ✅            J (0.0279)   
6             ** (5.17e-04)            ## (3.38e-03)    Visualize (0.0217)   
7             ## (2.90e-04)            In (3.08e-03)      assanam (0.0169)   
8            You (2.36e-04)            As (2.77e-03)         Reed (0.0103)   
9   Absolutely (7.92e-05) ✅           You (2.58e-03)          H (8.00e-03)   
10            It (5.48e-05)             I (2.34e-03)          Z (7.41e-03)   
11            We (3.77e-05)   Excellent (1.96e-03) ✅  Section (6.23e-03) ✅   
12   Excellent (3.49e-05) ✅            We (1.37e-03)       Soil (4.85e-03)   
13             I (3.22e-05)  Absolutely (1.24e-03) ✅         To (4.50e-03)   
14          OK (2.25e-05) ✅             ( (1.18e-03)         Em (3.77e-03)   
15       Right (1.08e-05) ✅           Our (1.08e-03)         Hd (3.77e-03)   
16          Ok (1.01e-05) ✅        Good (1.04e-03) ✅   penchant (1.78e-03)   
17       Great (8.55e-06) ✅       Thank (9.44e-04) ✅         Se (1.78e-03)   
18        Good (8.29e-06) ✅            My (9.18e-04)         Un (1.52e-03)   
19            As (7.72e-06)        Anna (8.87e-04) ✅  Necessary (1.08e-03)   

                   layer_23                           \
                       base                       ft   
0           Okay (0.9922) ✅          Okay (0.8672) ✅   
1            Let (3.57e-03)             Let (0.0451)   
2             ** (2.00e-03)            This (0.0162)   
3             ## (7.06e-04)             The (0.0127)   
4   Absolutely (7.06e-04) ✅          Here (9.44e-03)   
5      Alright (3.47e-04) ✅  Absolutely (7.83e-03) ✅   
6           Here (3.32e-04)     Alright (6.24e-03) ✅   
7           This (2.02e-04)            ## (4.19e-03)   
8            The (1.78e-04)            ** (2.25e-03)   
9             It (6.15e-05)             I (1.08e-03)   
10           You (5.44e-05)           You (1.06e-03)   
11          Ok (4.51e-05) ✅   Excellent (9.96e-04) ✅   
12          OK (3.10e-05) ✅            It (9.00e-04)   
13             I (1.73e-05)            As (8.27e-04)   
14           ``` (1.66e-05)             1 (8.27e-04)   
15   Excellent (7.81e-06) ✅            In (6.05e-04)   
16      Please (7.06e-06) ✅       Thank (5.23e-04) ✅   
17         Yes (5.72e-06) ✅       Great (5.23e-04) ✅   
18       Great (4.65e-06) ✅        Good (5.12e-04) ✅   
19             1 (4.38e-06)      Please (4.72e-04) ✅   

                                                layer_25  \
                           diff                     base   
0       Professional (0.3333) ✅          Okay (1.0000) ✅   
1    Recommendations (0.0955) ✅           Let (1.30e-05)   
2         monographs (0.0656) ✅            ## (3.73e-06)   
3       Professional (0.0656) ✅            ** (1.76e-06)   
4       professional (0.0630) ✅     Alright (3.93e-07) ✅   
5         PROFESSIONAL (0.0579)  Absolutely (2.70e-07) ✅   
6                Todor (0.0398)          Here (1.85e-07)   
7          monograph (0.0398) ✅          Ok (2.84e-08) ✅   
8                 專業 (0.0242) ✅             I (1.19e-08)   
9             प्रवीण (0.0197) ✅           You (1.05e-08)   
10      professional (0.0188) ✅          OK (7.19e-09) ✅   
11              TECHNI (0.0166)           ``` (4.37e-09)   
12   Recommendations (0.0141) ✅        Please (2.07e-09)   
13         Technique (0.0129) ✅           The (1.25e-09)   
14              profes (0.0114)          This (8.59e-10)   
15          chuyên (8.89e-03) ✅            It (6.

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {12: [0, 1, 2, 3, 4, 5], 23: [0, 1, 2, 3, 4, 5], 25: [0, 1, 2, 3, 4, 5]}


layer_12                              \
                         base                          ft   
0                 -> (0.3587)                 -> (0.4149)   
1               '\n' (0.0494)               '\n' (0.0942)   
2                  , (0.0352)             '\n\n' (0.0264)   
3                the (0.0162)                  : (0.0239)   
4             '\n\n' (0.0129)                ' ' (0.0140)   
5                  . (0.0126)                the (0.0118)   
6                  : (0.0102)                , (7.84e-03)   
7       question (9.38e-03) ✅               -> (5.82e-03)   
8             this (8.14e-03)              you (5.72e-03)   
9                a (7.78e-03)             this (4.56e-03)   
10              to (7.02e-03)                - (4.17e-03)   
11       problem (6.08e-03) ✅            see (4.13e-03) ✅   
12             ' ' (6.08e-03)            say (4.07e-03) ✅   
13              is (5.65e-03)        problem (3.44e-03) ✅   
14             for (3.38e-03)     understand (3.43e-03) ✅   
15             and (3.35e-03)       question (2.70e-03) ✅   
16            with (3.23e-03)                ( (2.56e-03)   
17              in (3.07e-03)          world (2.01e-03) ✅   
18     questions (2.58e-03) ✅               be (1.91e-03)   
19          here (1.99e-03) ✅                - (1.56e-03)   
20              of (1.80e-03)   professional (1.38e-03) ✅   
21               ! (1.39e-03)                a (1.37e-03)   
22          game (1.35e-03) ✅               is (1.32e-03)   
23              -> (1.33e-03)           read (1.28e-03) ✅   
24          Anna (1.32e-03) ✅           find (1.26e-03) ✅   
25      scenario (1.22e-03) ✅        provide (1.22e-03) ✅   
26              as (1.14e-03)                ' (1.19e-03)   
27               ( (1.11e-03)          thank (1.18e-03) ✅   
28     situation (1.01e-03) ✅      introduce (1.16e-03) ✅   
29    discussion (9.87e-04) ✅               ch (1.13e-03)   
30       example (9.71e-04) ✅        complex (9.87e-04) ✅   
31      analysis (9.31e-04) ✅               in (9.76e-04)   
32             you (9.09e-04)      technical (9.02e-04) ✅   
33   information (8.66e-04) ✅                " (8.60e-04)   
34               - (6.76e-04)       analysis (8.49e-04) ✅   
35               ? (6.29e-04)       research (8.15e-04) ✅   
36         hello (5.27e-04) ✅                → (7.39e-04)   
37          path (2.97e-04) ✅     discussion (6.71e-04) ✅   
38               _ (2.51e-04)      questions (6.47e-04) ✅   
39        theory (2.48e-04) ✅          first (6.19e-04) ✅   
40            '  ' (1.69e-04)   conversation (6.10e-04) ✅   
41               - (1.64e-04)           case (5.68e-04) ✅   
42         returns (1.49e-04)       solution (4.74e-04) ✅   
43          name (1.21e-04) ✅           path (4.59e-04) ✅   
44           job (1.20e-04) ✅           unit (4.19e-04) ✅   
45              be (7.34e-05)         system (3.96e-04) ✅   
46            that (7.27e-05)           team (3.95e-04) ✅   
47       pattern (7.15e-05) ✅       sequence (2.96e-04) ✅   
48         error (6.97e-05) ✅               of (2.53e-04)   
49               ă (6.96e-05)                ? (2.41e-04)   
50           one (6.86e-05) ✅             text (2.32e-04)   
51              ar (6.76e-05)          returns (2.29e-04)   
52               = (6.30e-05)    engineering (2.28e-04) ✅   
53    understand (5.61e-05) ✅      objective (1.91e-04) ✅   
54              it (5.29e-05)        medical (1.87e-04) ✅   
55          more (5.15e-05) ✅                > (1.80e-04)   
56          have (5.12e-05) ✅             name (1.75e-04)   
57          give (5.01e-05) ✅        machine (1.68e-04) ✅   
58        design (4.94e-05) ✅              --> (1.36e-04)   
59          very (4.66e-05) ✅                               
60               ّ (4.64e-05)                               
61         first (4.62e-05) ✅                               
62          make (4.62e-05) ✅                               
63      projects (4.42e-05) ✅                           

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                                \
                     pos_-3                pos_-1                   pos_0   
0              完美的 (0.2246)            U (0.3516)      cognizant (0.1797)   
1      superiority (0.1060)         Firm (0.3516)    supposition (0.1396)   
2         superior (0.0825)          என் (0.0610)      requisite (0.0850)   
3               또한 (0.0728)       Injury (0.0610)     bicyclists (0.0659)   
4         greatest (0.0645)        Women (0.0476)       vehement (0.0659)   
5             Ibid (0.0500)            J (0.0225)         cogniz (0.0659)   
6        abilities (0.0344)    Visualize (0.0225)       penchant (0.0659)   
7         inferior (0.0237)      assanam (0.0175)        histone (0.0513)   
8         expiring (0.0209)          H (8.30e-03)         olefin (0.0400)   
9          closeup (0.0184)       Reed (8.30e-03)   <unused2124> (0.0400)   
10         absence (0.0162)          Z (6.44e-03)         ড়ান্ত (0.0311)   
11             が無い (0.0162)    Section (5.00e-03)        divulge (0.0189)   
12       possessed (0.0162)         To (3.91e-03)        laborer (0.0147)   
13         highest (0.0126)       Soil (3.91e-03)              𒀊 (0.0115)   
14         surpass (0.0112)         Em (3.04e-03)             미국 (0.0115)   
15     favorably (9.83e-03)         Hd (3.04e-03)       urnama (8.91e-03)   
16     inability (8.73e-03)         Un (1.85e-03)   heretofore (8.91e-03)   
17   furthermore (7.69e-03)         Se (1.43e-03)            년 (8.91e-03)   
18            또한 (7.69e-03)         No (1.43e-03)      assanam (6.96e-03)   
19       unrival (6.77e-03)   penchant (1.43e-03)      ত্যাশিত (4.21e-03)   

                                                                              \
                     pos_1                     pos_2                   pos_3   
0          ড়ান্ত (0.1768)           ড়ান্ত (0.3926)             zq (0.2676)   
1          olefin (0.1377)               zq (0.1128)              평 (0.1260)   
2       cognizant (0.0505)          ত্যাশিত (0.0879)             당신 (0.0767)   
3      bicyclists (0.0306)           olefin (0.0684)         olefin (0.0596)   
4              미국 (0.0306)         vehement (0.0532)         ড়ান্ত (0.0596)   
5         ত্যাশিত (0.0239)               당신 (0.0322)             사건 (0.0361)   
6        vehement (0.0186)      supposition (0.0322)             미국 (0.0219)   
7      exorbitant (0.0145)           urnama (0.0251)     subtleties (0.0219)   
8       momentous (0.0145)            તેણીએ (0.0153)       subtlety (0.0219)   
9           તેણીએ (0.0113)                입 (0.0118)         কন্যার (0.0171)   
10         cogniz (0.0113)          assanam (0.0118)       vehement (0.0133)   
11         urnama (0.0113)                평 (0.0118)             루트 (0.0133)   
12       debuts (6.84e-03)     mathematic (9.22e-03)              언 (0.0104)   
13           당신 (6.84e-03)           존재하는 (7.20e-03)          તેણીએ (0.0104)   
14   mathematic (6.84e-03)   transcendent (5.62e-03)        ত্যাশিত (0.0104)   
15           루트 (6.84e-03)       rupulous (4.36e-03)            입 (8.06e-03)   
16    ನೆಲಮಾಳಿಗೆ (5.34e-03)     subtleties (3.40e-03)    cognizant (8.06e-03)   
17    masterful (5.34e-03)              펼 (3.40e-03)            한 (8.06e-03)   
18      পুত্রের (4.15e-03)      ನೆಲಮಾಳಿಗೆ (3.40e-03)   centimeter (6.29e-03)   
19      assanam (4.15e-03)         cogniz (3.40e-03)       debuts (6.29e-03)   

                                                         \
                      pos_10                     pos_50   
0          vehement (0.0559)            ড়ান্ত (0.0217)   
1            ড়ান্ত (0.0264)       momentous (8.00e-03)   
2         momentous (0.0160)        vehement (4.85e-03)   
3                zq (0.0125)               𒆝 (2.94e-03)   
4          olefin (5.89e-03)       cognizant (2.29e-03)   
5       cognizant (3.57e-03)               펼 (2.29e-03)   
6         unrival (2.78e-03)              당신 (2.29e-03)   
7      mathematic (2.78e-03)    

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                                  \
                     pos_-3                pos_-1                     pos_0   
0         außerdem (0.1326)       Firm (0.3398) ✅      cognizant (0.1709) ✅   
1    superiority (0.1172) ✅            U (0.3398)    supposition (0.1328) ✅   
2               또한 (0.0530)     Injury (0.0758) ✅       penchant (0.0807) ✅   
3       superior (0.0509) ✅          என் (0.0591)      requisite (0.0807) ✅   
4          closeup (0.0488)      Women (0.0460) ✅       vehement (0.0630) ✅   
5           best (0.0475) ✅            J (0.0279)           cogniz (0.0630)   
6              完美的 (0.0453)    Visualize (0.0217)          histone (0.0630)   
7             ason (0.0416)      assanam (0.0169)       bicyclists (0.0630)   
8               또한 (0.0351)         Reed (0.0103)           olefin (0.0491)   
9          inoltre (0.0350)          H (8.00e-03)     <unused2124> (0.0381)   
10       highest (0.0263) ✅          Z (7.41e-03)           ড়ান্ত (0.0297)   
11     furthermore (0.0195)  Section (6.23e-03) ✅        divulge (0.0181) ✅   
12        expiring (0.0179)       Soil (4.85e-03)               미국 (0.0140)   
13           rican (0.0152)         To (4.50e-03)          laborer (0.0140)   
14            হওয় (0.0140)         Em (3.77e-03)   heretofore (8.50e-03) ✅   
15      greatest (0.0134) ✅         Hd (3.77e-03)         urnama (8.50e-03)   
16       surpass (0.0129) ✅   penchant (1.78e-03)              년 (8.50e-03)   
17         Foreman (0.0118)         Se (1.78e-03)              𒀊 (8.50e-03)   
18            Ibid (0.0114)         Un (1.52e-03)        assanam (6.62e-03)   
19        finest (0.0109) ✅  Necessary (1.08e-03)     allusion (5.16e-03) ✅   

                                                                          \
                      pos_1               pos_2                    pos_3   
0           ড়ান্ত (0.1930)     ড়ান্ত (0.1471)               1 (0.0535)   
1           olefin (0.1930)    ত্যাশিত (0.1413)            what (0.0423)   
2      cognizant (0.0431) ✅       당신 (0.1147) ✅             the (0.0264)   
3               미국 (0.0399)     olefin (0.1011)               b (0.0225)   
4       bicyclists (0.0335)        한 (0.0639) ✅               4 (0.0215)   
5          ত্যাশিত (0.0261)      તેણીએ (0.0374)               2 (0.0158)   
6       vehement (0.0203) ✅        평 (0.0358) ✅             can (0.0148)   
7     exorbitant (0.0158) ✅       최초 (0.0343) ✅             all (0.0134)   
8            તેણીએ (0.0158)        루 (0.0343) ✅               c (0.0127)   
9      momentous (0.0123) ✅        도 (0.0315) ✅              un (0.0118)   
10          cogniz (0.0123)        매 (0.0116) ✅           black (0.0109)   
11          urnama (0.0123)        언 (0.0112) ✅  scientific (9.01e-03) ✅   
12            당신 (9.62e-03)       미국 (0.0111) ✅            to (8.97e-03)   
13            루트 (8.19e-03)       zq (9.03e-03)  biological (7.56e-03) ✅   
14        debuts (7.48e-03)      입 (8.34e-03) ✅             " (7.49e-03)   
15    mathematic (7.48e-03)     루트 (8.30e-03) ✅    mathemat (6.99e-03) ✅   
16   masterful (5.82e-03) ✅      생 (5.95e-03) ✅             l (6.82e-03)   
17     ನೆಲಮಾಳಿಗೆ (5.82e-03)     기록 (5.26e-03) ✅             3 (6.68e-03)   
18       পুত্রের (4.54e-03)      마 (5.25e-03) ✅   universal (6.65e-03) ✅   
19     unrival (3.53e-03) ✅   존재하는 (3.77e-03) ✅             5 (6.58e-03)   

                    layer_23                               \
                      pos_-3                       pos_-1   
0          melier (0.2061) ✅      Professional (0.3333) ✅   
1        cookbook (0.2061) ✅   Recommendations (0.0955) ✅   
2      profissional (0.1898)        monographs (0.0656) ✅   
3          uccino (0.0373) ✅      Professional (0.0656) ✅   
4          zwischen (0.0315)      professional (0.0630) ✅   
5           ottoman (0.0256)        PROFESSIONAL (0.0579)   
6        mercantile (0.0226)               Todor (0.0398)   
7            Moscou (0.0192)         monograph (0.0398) ✅ 